# Bronze Ingestion — Order Items

Loads the raw daily order-items CSVs from the e-commerce source volume into `ecommerce.bronze.bronze_order_items`, keeping raw values untouched:

* Source: 92 daily files in `order_items/landing/` (`order_items_YYYY-MM-DD.csv`), one per day covering 2025-08-01 through 2025-10-31 with no missing days
* Columns (from the first file's header): `dt`, `order_ts`, `customer_id`, `order_id`, `item_seq`, `product_id`, `quantity`, `unit_price_currency`, `unit_price`, `discount_pct`, `tax_amount`, `channel`, `coupon_code`

Two metadata columns are added:

* `source_file` — the full path of the source file the row came from (each row tracks its own daily file)
* `ingested_at` — the timestamp when the row was ingested

Verification below checks structural consistency across all 92 files (e.g. every row's `dt` matches a date pattern), so header inconsistencies would surface there rather than silently corrupting the load.

In [0]:
from pyspark.sql.functions import col, current_timestamp

source_path = "/Volumes/ecommerce/source_data/raw_data/ecomm-raw-data/order_items/landing/"
target_table = "ecommerce.bronze.bronze_order_items"

# Read all daily CSVs in the landing directory as-is (header row in each file)
df_order_items_raw = spark.read.csv(source_path, header=True, inferSchema=True)

# Attach ingestion metadata: full source file path and ingestion timestamp
df_bronze_order_items = df_order_items_raw.withColumn(
    "source_file", col("_metadata.file_path")
).withColumn(
    "ingested_at", current_timestamp()
)

# Idempotent full load into the bronze table
(
    df_bronze_order_items.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
# Structural checks across all source files (headers were only inspected for one file)
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT(DISTINCT source_file) AS distinct_source_files,
      COUNT_IF(CAST(dt AS STRING) IS NULL) AS null_dt,
      COUNT_IF(CAST(dt AS STRING) NOT RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$') AS bad_dt_rows,
      COUNT_IF(customer_id IS NULL) AS null_customer,
      COUNT_IF(order_id IS NULL) AS null_order,
      COUNT_IF(product_id IS NULL) AS null_product
    FROM ecommerce.bronze.bronze_order_items
""").show()

# Date coverage of the loaded data
spark.sql("""
    SELECT MIN(CAST(dt AS DATE)) AS first_dt, MAX(CAST(dt AS DATE)) AS last_dt,
           COUNT(DISTINCT CAST(dt AS DATE)) AS distinct_days
    FROM ecommerce.bronze.bronze_order_items
""").show()

display(spark.table("ecommerce.bronze.bronze_order_items").limit(10))